In [1]:
# ==========================================
# SOC Prediction using Heterogeneous Ensemble ML
# CORRECTED METHODOLOGY: Train on Current, Predict Future
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RepeatedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading data...")
# Load the dataset (the original small dataset that has both current and future climate variables)
data = pd.read_csv('Scenario.csv')

# Define predictors: Use CURRENT climate variables for training
predictors_current = ["TH_LAT", "Elev", "ec1", "EVI250", "Temp_Iberi", "slope250",
                      "twi", "Prec_World", "TH_LONG", "susm"]
target = "oc1"

X = data[predictors_current]
y = data[target]

print("\nStep 2: Splitting data into train and test (70-30)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# ==========================================
# 3. Metrics Helper Function
# ==========================================
def calculate_metrics(y_true, y_pred, model_name="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}\n")
    return rmse, mae, r2

# ==========================================
# 4. Model Training & Hyperparameter Tuning
# ==========================================
print("Step 3: Training models with CURRENT climate variables...")
cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=123)
models = {}
predictions = {}
metrics = {}

# 4.1 XGBoost
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(random_state=123, objective='reg:squarederror')
xgb_search = RandomizedSearchCV(xgb_model, {'n_estimators': [20, 30], 'max_depth': [5, 7], 'learning_rate': [0.01, 0.05]}, 
                                n_iter=5, scoring='neg_root_mean_squared_error', cv=cv, random_state=123, n_jobs=-1)
xgb_search.fit(X_train, y_train)
models['XGBoost'] = xgb_search.best_estimator_
predictions['XGBoost_train'] = models['XGBoost'].predict(X_train)
predictions['XGBoost_test'] = models['XGBoost'].predict(X_test)
metrics['XGBoost'] = calculate_metrics(y_test, predictions['XGBoost_test'], "XGBoost")

# 4.2 Random Forest
print("Training Random Forest...")
rf_model = RandomForestRegressor(random_state=123)
rf_search = RandomizedSearchCV(rf_model, {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}, 
                               n_iter=5, scoring='neg_root_mean_squared_error', cv=cv, random_state=123, n_jobs=-1)
rf_search.fit(X_train, y_train)
models['Random Forest'] = rf_search.best_estimator_
predictions['RF_train'] = models['Random Forest'].predict(X_train)
predictions['RF_test'] = models['Random Forest'].predict(X_test)
metrics['Random Forest'] = calculate_metrics(y_test, predictions['RF_test'], "Random Forest")

# 4.3 SVR
print("Training SVR...")
svr_pipeline = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
svr_search = RandomizedSearchCV(svr_pipeline, {'svr__C': [2**i for i in range(-5, 5)], 'svr__gamma': [2**i for i in range(-5, 5)]}, 
                                n_iter=10, scoring='neg_root_mean_squared_error', cv=cv, random_state=123, n_jobs=-1)
svr_search.fit(X_train, y_train)
models['SVR'] = svr_search.best_estimator_
predictions['SVM_train'] = models['SVR'].predict(X_train)
predictions['SVM_test'] = models['SVR'].predict(X_test)
metrics['SVR'] = calculate_metrics(y_test, predictions['SVM_test'], "SVR")

# 4.4 Decision Tree
print("Training Decision Tree...")
dt_model = DecisionTreeRegressor(random_state=123)
dt_search = RandomizedSearchCV(dt_model, {'max_depth': [5, 10, 20], 'min_samples_split': [2, 5]}, 
                               n_iter=5, scoring='neg_root_mean_squared_error', cv=cv, random_state=123, n_jobs=-1)
dt_search.fit(X_train, y_train)
models['Decision Tree'] = dt_search.best_estimator_
predictions['DT_train'] = models['Decision Tree'].predict(X_train)
predictions['DT_test'] = models['Decision Tree'].predict(X_test)
metrics['Decision Tree'] = calculate_metrics(y_test, predictions['DT_test'], "Decision Tree")

# 4.5 Cubist (Proxy: HistGradientBoostingRegressor)
print("Training Cubist (Proxy)...")
cubist_model = HistGradientBoostingRegressor(random_state=123)
cubist_search = RandomizedSearchCV(cubist_model, {'learning_rate': [0.01, 0.05], 'max_iter': [100, 200]}, 
                                   n_iter=5, scoring='neg_root_mean_squared_error', cv=cv, random_state=123, n_jobs=-1)
cubist_search.fit(X_train, y_train)
models['Cubist'] = cubist_search.best_estimator_
predictions['Cubist_train'] = models['Cubist'].predict(X_train)
predictions['Cubist_test'] = models['Cubist'].predict(X_test)
metrics['Cubist'] = calculate_metrics(y_test, predictions['Cubist_test'], "Cubist")

# ==========================================
# 5. Ensemble Modeling (1000 Iterations)
# ==========================================
print("Step 4: Building Ensemble model (1000 iterations)...")
ensemble_input_test = pd.DataFrame({
    'RF': predictions['RF_test'], 'DT': predictions['DT_test'], 'XGBoost': predictions['XGBoost_test'],
    'Cubist': predictions['Cubist_test'], 'SVM': predictions['SVM_test']
})

np.random.seed(123)
best_R2, best_RMSE, best_MAE, best_weights = -np.inf, np.inf, np.inf, None

for i in range(1000):
    if (i + 1) % 200 == 0:
        print(f"   ... iteration {i+1} completed ...")
    weights = np.random.uniform(0, 10, 5)
    weights = weights / weights.sum()
    ensemble_pred = (ensemble_input_test['RF'] * weights[0] + ensemble_input_test['DT'] * weights[1] +
                     ensemble_input_test['XGBoost'] * weights[2] + ensemble_input_test['Cubist'] * weights[3] +
                     ensemble_input_test['SVM'] * weights[4])
    
    r2 = r2_score(y_test, ensemble_pred)
    if r2 > best_R2:
        best_R2, best_RMSE, best_MAE = r2, np.sqrt(mean_squared_error(y_test, ensemble_pred)), mean_absolute_error(y_test, ensemble_pred)
        best_weights = weights

print("\nEnsemble building finished!")
print("--- Best Ensemble Results (Corrected Methodology) ---")
print(f"Best RMSE: {best_RMSE:.4f}")
print(f"Best MAE:  {best_MAE:.4f}")
print(f"Best R2:   {best_R2:.4f}\n")
print("Best Weights Found (Normalized):")
print(f"RF: {best_weights[0]:.4f}, DT: {best_weights[1]:.4f}, XGBoost: {best_weights[2]:.4f}, Cubist: {best_weights[3]:.4f}, SVM: {best_weights[4]:.4f}")

# ==========================================
# 6. FUTURE PREDICTION (Using Scenario Climate Variables)
# ==========================================
print("\nStep 5: Predicting FUTURE SOC using SCENARIO climate variables...")

# Prepare the future dataframe
# We start with the current predictors structure, then swap the climate variables
future_pred_df = data[predictors_current].copy()

# SWAP: Replace current climate with future scenario data
# Note: t58560 = future temperature, pr58060 = future precipitation
future_pred_df['Temp_Iberi'] = data['t58560']
future_pred_df['Prec_World'] = data['pr58060']

# Now predict using the trained models
future_predictions = pd.DataFrame()
future_predictions['RF'] = models['Random Forest'].predict(future_pred_df)
future_predictions['DT'] = models['Decision Tree'].predict(future_pred_df)
future_predictions['XGBoost'] = models['XGBoost'].predict(future_pred_df)
future_predictions['Cubist'] = models['Cubist'].predict(future_pred_df)
future_predictions['SVM'] = models['SVR'].predict(future_pred_df)

# Apply the best weights found during ensemble training
future_predictions['Ensemble_Prediction'] = (
    future_predictions['RF'] * best_weights[0] +
    future_predictions['DT']* best_weights[1] +
    future_predictions['XGBoost'] * best_weights[2] +
    future_predictions['Cubist'] * best_weights[3] +
    future_predictions['SVM'] * best_weights[4]
)

if 'POINTID' in data.columns:
    future_predictions['POINTID'] = data['POINTID']

# Save the final output
future_predictions.to_csv('scenario_predictions_future_corrected.csv', index=False)
print("Corrected future predictions saved to 'scenario_predictions_future_corrected.csv'.")
print("All steps completed successfully!")

Step 1: Loading data...

Step 2: Splitting data into train and test (70-30)...
Train shape: (856, 10), Test shape: (367, 10)
Step 3: Training models with CURRENT climate variables...
Training XGBoost...
--- XGBoost ---
RMSE: 112.0635, MAE: 87.1641, R2: 0.2909

Training Random Forest...
--- Random Forest ---
RMSE: 106.9096, MAE: 81.0984, R2: 0.3546

Training SVR...
--- SVR ---
RMSE: 116.8712, MAE: 85.1391, R2: 0.2287

Training Decision Tree...
--- Decision Tree ---
RMSE: 122.4401, MAE: 92.4979, R2: 0.1535

Training Cubist (Proxy)...
--- Cubist ---
RMSE: 107.2502, MAE: 83.5135, R2: 0.3505

Step 4: Building Ensemble model (1000 iterations)...
   ... iteration 200 completed ...
   ... iteration 400 completed ...
   ... iteration 600 completed ...
   ... iteration 800 completed ...
   ... iteration 1000 completed ...

Ensemble building finished!
--- Best Ensemble Results (Corrected Methodology) ---
Best RMSE: 106.4209
Best MAE:  80.9252
Best R2:   0.3605

Best Weights Found (Normalized):
RF

In [3]:
# ==========================================
# 7. CURRENT PREDICTION (Using Current Climate Variables)
# ==========================================
print("Step 6: Predicting CURRENT SOC using CURRENT climate variables...")

# Prepare the current dataframe (using the same predictors we trained on)
current_pred_df = data[predictors_current].copy()

# Predict using the trained base models
current_predictions = pd.DataFrame()
current_predictions['RF'] = models['Random Forest'].predict(current_pred_df)
current_predictions['DT'] = models['Decision Tree'].predict(current_pred_df)
current_predictions['XGBoost'] = models['XGBoost'].predict(current_pred_df)
current_predictions['Cubist'] = models['Cubist'].predict(current_pred_df)
current_predictions['SVM'] = models['SVR'].predict(current_pred_df)

# Apply the best weights found during ensemble training
current_predictions['Ensemble_Prediction'] = (
    current_predictions['RF'] * best_weights[0] +
    current_predictions['DT'] * best_weights[1] +
    current_predictions['XGBoost'] * best_weights[2] +
    current_predictions['Cubist'] * best_weights[3] +
    current_predictions['SVM'] * best_weights[4]
)

if 'POINTID' in data.columns:
    current_predictions['POINTID'] = data['POINTID']

# Save the final output
current_predictions.to_csv('current_predictions_ensemble.csv', index=False)
print("Current SOC predictions saved to 'current_predictions_ensemble.csv'.")
print("All steps completed successfully!")

Step 6: Predicting CURRENT SOC using CURRENT climate variables...
Current SOC predictions saved to 'current_predictions_ensemble.csv'.
All steps completed successfully!


In [5]:
# Save the optimized weights file
weights_df = pd.DataFrame({
    'Model': ['RF', 'DT', 'XGBoost', 'Cubist', 'SVM'],
    'Weight': best_weights
})
weights_df.to_csv('ensemble_weights_corrected.csv', index=False)
print("Corrected ensemble weights saved to 'ensemble_weights_corrected.csv'.")

Corrected ensemble weights saved to 'ensemble_weights_corrected.csv'.
